# Example how to read modify and publish content using the Confluence API


## Imports and setup

In [1]:
import json
import requests

with open('config.json') as f:
    config = json.load(f)

assert len(config['apiurl']) > 0

authentication = (config['username'], config['password'])

(config['apiurl'], config['space'], config['page'])

('https://foryouandyourteam.com/rest/api/content',
 '~bue',
 'Automation test page')

In [2]:
import xml.dom.minidom
import IPython

def beautify(flat_xml):
    dom = xml.dom.minidom.parseString('<enclosing-content>{}</enclosing-content>'.format(flat_xml))
    pretty_xml_as_string = dom.toprettyxml()
    return pretty_xml_as_string

# Read the page content

In [3]:
resp = requests.get(config['apiurl'], auth=authentication, params= {
               'title': config['page'],
               'spaceKey': config['space'],
               'expand': 'body.view,version'
            }
        )
resp.raise_for_status()
results = resp.json().get('results')
results

[{'id': '139396216',
  'type': 'page',
  'status': 'current',
  'title': 'Automation test page',
  'version': {'by': {'type': 'known',
    'username': 'bue',
    'userKey': '8a44818c734d6ea10175079ad55f0036',
    'profilePicture': {'path': '/download/attachments/132335148/user-avatar',
     'width': 48,
     'height': 48,
     'isDefault': False},
    'displayName': 'Christian Bühlmann',
    '_links': {'self': 'https://foryouandyourteam.com/rest/api/user?key=8a44818c734d6ea10175079ad55f0036'},
    '_expandable': {'status': ''}},
   'when': '2020-12-18T10:26:23.000+01:00',
   'message': 'Bla',
   'number': 6,
   'minorEdit': False,
   'hidden': False,
   '_links': {'self': 'https://foryouandyourteam.com/rest/experimental/content/139396216/version/6'},
   '_expandable': {'content': '/rest/api/content/139396216'}},
  'body': {'view': {'value': '<p>A</p>....',
    'representation': 'storage',
    '_expandable': {'webresource': '',
     'content': '/rest/api/content/139396216'}},
   '_expan

Show current content with syntax higlighted HTML

In [4]:
page = results[0]
current_content = page['body']['view']['value']
IPython.display.Code(beautify(current_content))

<?xml version="1.0" ?>
<enclosing-content>
	<p>A</p>
	....
</enclosing-content>

# Apply changes

In [5]:
current_version = page['version']['number']
version = current_version + 1
change_message = 'Bla'.format(version)

content = current_content + '.' #current_content + '<h2>Version {}</h2><p>Modified from {}</p>'.format(version, 'bla') 

In [6]:
IPython.display.Code(beautify(content))

<?xml version="1.0" ?>
<enclosing-content>
	<p>A</p>
	.....
</enclosing-content>

Beware of the version field format

In [7]:
d = {
    'title': page['title'],
    'id': page['id'],
    'type': page['type'],
    'version': { 'number': version, 'notifyEdit': False, 'message': change_message },
    'body': {
        'storage': {
            'representation': 'storage',
            'value': content
        }
    }
}

json.dumps(d)

'{"title": "Automation test page", "id": "139396216", "type": "page", "version": {"number": 7, "notifyEdit": false, "message": "Bla"}, "body": {"storage": {"representation": "storage", "value": "<p>A</p>....."}}}'

## Publish the updated content

*HTTPError: 409 Client Error* indicates, either version or content are invalid

In [10]:
url = page['_links']['self']

put_response = requests.put(url, headers={'Content-Type': 'application/json'}, data=json.dumps(d), auth=authentication)

# error reason is hidden in response object: therefore dump it
if not put_response.status_code == requests.codes.ok:
    from pprint import pprint
    from urllib.error import HTTPError
    pprint(vars(put_response))
    raise HTTPError(url, put_response.status_code, put_response.json(), put_response.headers, None)



{'_content': b'{"statusCode":409,"data":{"authorized":false,"valid":true,"allow'
             b'edInReadOnlyMode":true,"errors":[],"successful":false},"message"'
             b':"Version must be incremented on update. Current version is: 7",'
             b'"reason":"Conflict"}',
 '_content_consumed': True,
 '_next': None,
 'connection': <requests.adapters.HTTPAdapter object at 0x7fa1f7a32610>,
 'cookies': <RequestsCookieJar[Cookie(version=0, name='JSESSIONID', value='4602C465D54BE6F2EDC3C4FEBC482B98', port=None, port_specified=False, domain='foryouandyourteam.com', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=True, expires=None, discard=True, comment=None, comment_url=None, rest={'HttpOnly': None}, rfc2109=False)]>,
 'elapsed': datetime.timedelta(microseconds=189901),
 'encoding': None,
 'headers': {'Date': 'Fri, 18 Dec 2020 09:27:17 GMT', 'Server': 'Apache/2.4.6 (CentOS)', 'X-ASEN': 'SEN-2084351', 'X-Seraph-LoginReason': 'OK', 'X-AUSERNAME':

HTTPError: HTTP Error 409: {'statusCode': 409, 'data': {'authorized': False, 'valid': True, 'allowedInReadOnlyMode': True, 'errors': [], 'successful': False}, 'message': 'Version must be incremented on update. Current version is: 7', 'reason': 'Conflict'}